In [1]:
import pandas as pd
import numpy as np
import random
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

# ==========================================
# 1. LOAD DATA & TRAIN THE AI MODEL
# ==========================================
# Load the CSV (make sure the file is in the same folder as this notebook)
DeepSeek_df = pd.read_csv("DeepSeek_df_clean.csv")

FEATURES = [
    'TyreLife', 'TyreLifePercent', 'Avg_TrackTemp', 'Avg_AirTemp', 
    'TrackTemp_Delta', 'Peak_TyreEnergy', 'Energy_Std', 
    'PaceTrend', 'Quad_Term', 'DegAcceleration', 
    'Compound', 'CircuitArchetype', 'StintPhase'
]
TARGET = 'T2_Deg_Target'

train_df = DeepSeek_df.dropna(subset=[TARGET])
X = train_df[FEATURES]
y = train_df[TARGET]

cat_features = ['Compound', 'CircuitArchetype', 'StintPhase']
num_features = [c for c in FEATURES if c not in cat_features]

preprocessor = ColumnTransformer([
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_features),
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_features)
])

model = Pipeline([
    ("prep", preprocessor), 
    ("rf", RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1))
])
model.fit(X, y)
print("✅ AI Model Trained!")

# ==========================================
# 2. THE SIMULATOR ENGINE
# ==========================================
def simulate_full_race(race_name, driver_code, rival_code, target_pit_lap, rival_pit_lap, pit_loss=22.0):
    my_data = DeepSeek_df[(DeepSeek_df['EventName'] == race_name) & (DeepSeek_df['Driver'] == driver_code)].copy()
    rival_data = DeepSeek_df[(DeepSeek_df['EventName'] == race_name) & (DeepSeek_df['Driver'] == rival_code)].copy()
    
    if my_data.empty or rival_data.empty: return None
    total_laps = int(max(my_data['LapNumber'].max(), rival_data['LapNumber'].max()))
    
    all_laps = pd.DataFrame({'LapNumber': range(1, total_laps + 1)})
    my_data = all_laps.merge(my_data, on='LapNumber', how='left').ffill()
    rival_data = all_laps.merge(rival_data, on='LapNumber', how='left').ffill()

    my_base_pace = my_data['LapTimeSeconds'].min()
    rival_base_pace = rival_data['LapTimeSeconds'].min()
    
    my_time, rival_time = 0.0, 0.0
    my_wear, rival_wear = 0.0, 0.0
    my_tyre_age, rival_tyre_age = 0, 0
    gap = 1.0 
    
    safety_car_lap = random.randint(10, total_laps) if random.random() < 0.20 else None
    
    for lap in range(1, total_laps + 1):
        my_row = my_data[my_data['LapNumber'] == lap].iloc[0]
        my_features = pd.DataFrame([{
            'TyreLife': my_tyre_age, 'TyreLifePercent': my_tyre_age / 25.0,
            'Avg_TrackTemp': my_row['Avg_TrackTemp'], 'Avg_AirTemp': my_row['Avg_AirTemp'],
            'TrackTemp_Delta': my_row['TrackTemp_Delta'], 'Peak_TyreEnergy': my_row['Peak_TyreEnergy'],
            'Energy_Std': my_row['Energy_Std'], 'PaceTrend': my_row['PaceTrend'],
            'Quad_Term': my_row['Quad_Term'], 'DegAcceleration': my_row['DegAcceleration'],
            'Compound': my_row['Compound'], 'CircuitArchetype': my_row['CircuitArchetype'],
            'StintPhase': 'Early' if my_tyre_age <= 5 else ('Mid' if my_tyre_age <= 15 else 'Late')
        }])
        my_wear_next = max(0, model.predict(my_features)[0])
        my_lap_time = my_base_pace + my_wear

        rival_row = rival_data[rival_data['LapNumber'] == lap].iloc[0]
        rival_features = pd.DataFrame([{
            'TyreLife': rival_tyre_age, 'TyreLifePercent': rival_tyre_age / 25.0,
            'Avg_TrackTemp': rival_row['Avg_TrackTemp'], 'Avg_AirTemp': rival_row['Avg_AirTemp'],
            'TrackTemp_Delta': rival_row['TrackTemp_Delta'], 'Peak_TyreEnergy': rival_row['Peak_TyreEnergy'],
            'Energy_Std': rival_row['Energy_Std'], 'PaceTrend': rival_row['PaceTrend'],
            'Quad_Term': rival_row['Quad_Term'], 'DegAcceleration': rival_row['DegAcceleration'],
            'Compound': rival_row['Compound'], 'CircuitArchetype': rival_row['CircuitArchetype'],
            'StintPhase': 'Early' if rival_tyre_age <= 5 else ('Mid' if rival_tyre_age <= 15 else 'Late')
        }])
        rival_wear_next = max(0, model.predict(rival_features)[0])
        rival_lap_time = rival_base_pace + rival_wear
        
        if lap == rival_pit_lap:
            rival_lap_time += pit_loss
            rival_tyre_age = 1
            rival_wear = 0.0
        else:
            rival_tyre_age += 1
            rival_wear += rival_wear_next

        actual_pit_loss = pit_loss
        if safety_car_lap and lap == safety_car_lap:
            actual_pit_loss = 10.0 
            
        if lap == target_pit_lap:
            my_lap_time += actual_pit_loss
            my_tyre_age = 1
            my_wear = 0.0
        else:
            my_tyre_age += 1
            my_wear += my_wear_next
            
        if 0 < (gap + (my_lap_time - rival_lap_time)) < 1.5:
            my_lap_time += 0.3 
            
        my_time += my_lap_time
        rival_time += rival_lap_time
        gap = rival_time - my_time 

    return my_time, rival_time, gap

# ==========================================
# 3. H1: THE OBJECTIVE FUNCTION & TEST
# ==========================================
def calculate_points(gap):
    if gap > 0: return 25, "P1" 
    else: return 18, "P2" 

RACE = "Bahrain Grand Prix"
ME = "HAM"
RIVAL = "VER"

print("\n🔄 Running H1 Optimization (Maximizing Points)...\n")

# Test pitting on Lap 24 (Undercut), 25 (Same), and 26 (Overcut)
for my_pit in [24, 25, 26]:
    my_time, rival_time, gap = simulate_full_race(RACE, ME, RIVAL, target_pit_lap=my_pit, rival_pit_lap=25)
    
    points, position = calculate_points(gap)
    
    print(f"Strategy: Pit on Lap {my_pit}")
    print(f"Result: We finish {position}. Points: {points}")
    if gap > 0:
        print(f"   (We beat VER by {abs(gap):.2f} seconds)\n")
    else:
        print(f"   (We lost to VER by {abs(gap):.2f} seconds)\n")

✅ AI Model Trained!

🔄 Running H1 Optimization (Maximizing Points)...

Strategy: Pit on Lap 24
Result: We finish P2. Points: 18
   (We lost to VER by 5.94 seconds)

Strategy: Pit on Lap 25
Result: We finish P2. Points: 18
   (We lost to VER by 4.58 seconds)

Strategy: Pit on Lap 26
Result: We finish P2. Points: 18
   (We lost to VER by 4.26 seconds)



# ----------- UNDERCUT / OVERCUT ENGINE ----------------

In [2]:
# ==========================================
# H2: THE UNDERCUT / OVERCUT ENGINE
# ==========================================
def test_undercut_overcut(race_name, me, rival, rival_pit_lap):
    print(f"🏎️ Testing Reactions: Rival ({rival}) pits on Lap {rival_pit_lap}")
    print("-" * 50)
    
    # We will test 3 scenarios against the rival
    scenarios = {
        "Undercut (Pit 1 lap early)": rival_pit_lap - 1,
        "Same Strategy (Pit same lap)": rival_pit_lap,
        "Overcut (Pit 1 lap late)": rival_pit_lap + 1
    }
    
    best_action = ""
    best_gap = -999
    
    for action_name, my_pit_lap in scenarios.items():
        # Run the simulator
        result = simulate_full_race(race_name, me, rival, target_pit_lap=my_pit_lap, rival_pit_lap=rival_pit_lap)
        
        if result:
            
            my_time, rival_time, gap = result
            points, position = calculate_points(gap)
            
            if gap > 0:
                outcome = f"WIN by {abs(gap):.2f}s ({position})"
            else:
                outcome = f"LOSE by {abs(gap):.2f}s ({position})"
                
            print(f"Action: {action_name.ljust(30)} -> {outcome}")
            
            # Track the best action
            if gap > best_gap:
                best_gap = gap
                best_action = action_name
                
    print("-" * 50)
    print(f"🏆 BEST COUNTER-MOVE: {best_action}\n")

# Run H2
RACE = "Bahrain Grand Prix"
ME = "HAM"
RIVAL = "VER"

# Scenario: Verstappen pits on Lap 25. How do we react?
test_undercut_overcut(RACE, ME, RIVAL, rival_pit_lap=25)

# Scenario: What if Verstappen pits super early on Lap 15?
test_undercut_overcut(RACE, ME, RIVAL, rival_pit_lap=15)

🏎️ Testing Reactions: Rival (VER) pits on Lap 25
--------------------------------------------------
Action: Undercut (Pit 1 lap early)     -> LOSE by 5.94s (P2)
Action: Same Strategy (Pit same lap)   -> LOSE by 4.58s (P2)
Action: Overcut (Pit 1 lap late)       -> LOSE by 4.26s (P2)
--------------------------------------------------
🏆 BEST COUNTER-MOVE: Overcut (Pit 1 lap late)

🏎️ Testing Reactions: Rival (VER) pits on Lap 15
--------------------------------------------------
Action: Undercut (Pit 1 lap early)     -> LOSE by 7.75s (P2)
Action: Same Strategy (Pit same lap)   -> LOSE by 5.40s (P2)
Action: Overcut (Pit 1 lap late)       -> LOSE by 3.62s (P2)
--------------------------------------------------
🏆 BEST COUNTER-MOVE: Overcut (Pit 1 lap late)



# -------------------- H3: The Monte Carlo Engine (Probability & Risk) --------------------

In [3]:
# ==========================================
# H3: THE MONTE CARLO ENGINE (Probability)
# ==========================================
def monte_carlo_simulation(race_name, me, rival, my_pit_lap, rival_pit_lap, num_simulations=100):
    print(f"🎲 Running {num_simulations} simulations for Strategy: Pit on Lap {my_pit_lap}...")
    
    wins = 0
    losses = 0
    total_gap = 0
    
    for i in range(num_simulations):
        # Run the race (Safety car is random each time)
        result = simulate_full_race(race_name, me, rival, target_pit_lap=my_pit_lap, rival_pit_lap=rival_pit_lap)
        
        if result:
            my_time, rival_time, gap = result
            total_gap += gap
            
            if gap > 0:
                wins += 1
            else:
                losses += 1
                
    # Calculate probabilities
    win_prob = (wins / num_simulations) * 100
    loss_prob = (losses / num_simulations) * 100
    avg_gap = total_gap / num_simulations
    
    return win_prob, loss_prob, avg_gap

# ==========================================
# RUNNING THE H3 TEST
# ==========================================
RACE = "Bahrain Grand Prix"
ME = "HAM"
RIVAL = "VER"
RIVAL_PIT = 25

# Test our 3 strategies 100 times each
print("--- UNDERCUT (Pit Lap 24) ---")
win, loss, gap = monte_carlo_simulation(RACE, ME, RIVAL, my_pit_lap=24, rival_pit_lap=RIVAL_PIT, num_simulations=100)
print(f"Win Rate: {win}% | Loss Rate: {loss}% | Avg Gap: {gap:.2f}s\n")

print("--- SAME STRATEGY (Pit Lap 25) ---")
win, loss, gap = monte_carlo_simulation(RACE, ME, RIVAL, my_pit_lap=25, rival_pit_lap=RIVAL_PIT, num_simulations=100)
print(f"Win Rate: {win}% | Loss Rate: {loss}% | Avg Gap: {gap:.2f}s\n")

print("--- OVERCUT (Pit Lap 26) ---")
win, loss, gap = monte_carlo_simulation(RACE, ME, RIVAL, my_pit_lap=26, rival_pit_lap=RIVAL_PIT, num_simulations=100)
print(f"Win Rate: {win}% | Loss Rate: {loss}% | Avg Gap: {gap:.2f}s\n")

--- UNDERCUT (Pit Lap 24) ---
🎲 Running 100 simulations for Strategy: Pit on Lap 24...
Win Rate: 2.0% | Loss Rate: 98.0% | Avg Gap: -5.68s

--- SAME STRATEGY (Pit Lap 25) ---
🎲 Running 100 simulations for Strategy: Pit on Lap 25...
Win Rate: 0.0% | Loss Rate: 100.0% | Avg Gap: -4.58s

--- OVERCUT (Pit Lap 26) ---
🎲 Running 100 simulations for Strategy: Pit on Lap 26...
Win Rate: 1.0% | Loss Rate: 99.0% | Avg Gap: -4.14s



# -------------------- H4: The Track Difficulty & Traffic Trap --------------------


In [4]:
def simulate_full_race(race_name, driver_code, rival_code, target_pit_lap, rival_pit_lap, pit_loss=22.0):
    my_data = DeepSeek_df[(DeepSeek_df['EventName'] == race_name) & (DeepSeek_df['Driver'] == driver_code)].copy()
    rival_data = DeepSeek_df[(DeepSeek_df['EventName'] == race_name) & (DeepSeek_df['Driver'] == rival_code)].copy()
    
    if my_data.empty or rival_data.empty: return None
    total_laps = int(max(my_data['LapNumber'].max(), rival_data['LapNumber'].max()))
    
    all_laps = pd.DataFrame({'LapNumber': range(1, total_laps + 1)})
    my_data = all_laps.merge(my_data, on='LapNumber', how='left').ffill()
    rival_data = all_laps.merge(rival_data, on='LapNumber', how='left').ffill()

    my_base_pace = my_data['LapTimeSeconds'].min()
    rival_base_pace = rival_data['LapTimeSeconds'].min()
    
    my_time, rival_time = 0.0, 0.0
    my_wear, rival_wear = 0.0, 0.0
    my_tyre_age, rival_tyre_age = 0, 0
    gap = 1.0 
    
    safety_car_lap = random.randint(10, total_laps) if random.random() < 0.20 else None
    
    # ==========================================
    # H4: THE TRACK DIFFICULTY MULTIPLIER
    # 3.0 = Very hard to pass (Monaco). 
    # 1.0 = Normal (Bahrain).
    # 0.5 = Easy to pass (Monza/China).
    # ==========================================
    if "Monaco" in race_name:
        track_multiplier = 3.0
    elif "Italy" in race_name or "China" in race_name:
        track_multiplier = 0.5
    else:
        track_multiplier = 1.0

    for lap in range(1, total_laps + 1):
        my_row = my_data[my_data['LapNumber'] == lap].iloc[0]
        my_features = pd.DataFrame([{
            'TyreLife': my_tyre_age, 'TyreLifePercent': my_tyre_age / 25.0,
            'Avg_TrackTemp': my_row['Avg_TrackTemp'], 'Avg_AirTemp': my_row['Avg_AirTemp'],
            'TrackTemp_Delta': my_row['TrackTemp_Delta'], 'Peak_TyreEnergy': my_row['Peak_TyreEnergy'],
            'Energy_Std': my_row['Energy_Std'], 'PaceTrend': my_row['PaceTrend'],
            'Quad_Term': my_row['Quad_Term'], 'DegAcceleration': my_row['DegAcceleration'],
            'Compound': my_row['Compound'], 'CircuitArchetype': my_row['CircuitArchetype'],
            'StintPhase': 'Early' if my_tyre_age <= 5 else ('Mid' if my_tyre_age <= 15 else 'Late')
        }])
        my_wear_next = max(0, model.predict(my_features)[0])
        my_lap_time = my_base_pace + my_wear

        rival_row = rival_data[rival_data['LapNumber'] == lap].iloc[0]
        rival_features = pd.DataFrame([{
            'TyreLife': rival_tyre_age, 'TyreLifePercent': rival_tyre_age / 25.0,
            'Avg_TrackTemp': rival_row['Avg_TrackTemp'], 'Avg_AirTemp': rival_row['Avg_AirTemp'],
            'TrackTemp_Delta': rival_row['TrackTemp_Delta'], 'Peak_TyreEnergy': rival_row['Peak_TyreEnergy'],
            'Energy_Std': rival_row['Energy_Std'], 'PaceTrend': rival_row['PaceTrend'],
            'Quad_Term': rival_row['Quad_Term'], 'DegAcceleration': rival_row['DegAcceleration'],
            'Compound': rival_row['Compound'], 'CircuitArchetype': rival_row['CircuitArchetype'],
            'StintPhase': 'Early' if rival_tyre_age <= 5 else ('Mid' if rival_tyre_age <= 15 else 'Late')
        }])
        rival_wear_next = max(0, model.predict(rival_features)[0])
        rival_lap_time = rival_base_pace + rival_wear
        
        if lap == rival_pit_lap:
            rival_lap_time += pit_loss
            rival_tyre_age = 1
            rival_wear = 0.0
        else:
            rival_tyre_age += 1
            rival_wear += rival_wear_next

        actual_pit_loss = pit_loss
        if safety_car_lap and lap == safety_car_lap:
            actual_pit_loss = 10.0 
            
        if lap == target_pit_lap:
            my_lap_time += actual_pit_loss
            my_tyre_age = 1
            my_wear = 0.0
        else:
            my_tyre_age += 1
            my_wear += my_wear_next
            
        # ==========================================
        # H4: THE DIRTY AIR TRAP
        # If we are within 1.5s, we lose time. The penalty is multiplied by the track difficulty.
        # ==========================================
        if 0 < gap < 1.5:
            dirty_air_penalty = 0.3 * track_multiplier
            my_lap_time += dirty_air_penalty 
            
        my_time += my_lap_time
        rival_time += rival_lap_time
        gap = rival_time - my_time 

    return my_time, rival_time, gap

In [5]:
# Run H4 Test: PIA vs VER

print("=========================================")
print("🏁 TESTING MONACO (Hard to pass, Multiplier 3.0)")
print("=========================================")
# Assume VER pits on Lap 30. We try to undercut on Lap 29.
result_monaco = simulate_full_race("Monaco Grand Prix", "PIA", "VER", target_pit_lap=29, rival_pit_lap=30)
my_time, rival_time, gap = result_monaco
if gap > 0:
    print(f"Result: PIA WINS by {abs(gap):.2f}s (Fresh tyres worked!)")
else:
    print(f"Result: PIA LOSES by {abs(gap):.2f}s (Got stuck in Dirty Air Trap!)")

print("\n=========================================")
print("🏁 TESTING CHINA (Easy to pass, Multiplier 0.5)")
print("=========================================")
# Same strategy: VER pits on Lap 30, we pit on Lap 29.
result_china = simulate_full_race("Chinese Grand Prix", "PIA", "VER", target_pit_lap=29, rival_pit_lap=30)
my_time, rival_time, gap = result_china
if gap > 0:
    print(f"Result: PIA WINS by {abs(gap):.2f}s (Fresh tyres worked!)")
else:
    print(f"Result: PIA LOSES by {abs(gap):.2f}s (Got stuck in Dirty Air Trap!)")

🏁 TESTING MONACO (Hard to pass, Multiplier 3.0)
Result: PIA WINS by 30.08s (Fresh tyres worked!)

🏁 TESTING CHINA (Easy to pass, Multiplier 0.5)
Result: PIA WINS by 2.69s (Fresh tyres worked!)


In [6]:
import random

# Random Test: LEC vs NOR at Silverstone
RACE = "British Grand Prix"
ME = "LEC"
RIVAL = "NOR"
MY_PIT = 28     # LEC tries the Overcut (pitting later)
RIVAL_PIT = 22  # NOR pits early

print(f"🏁 Testing {ME} vs {RIVAL} at {RACE}")
print(f"Strategy: {RIVAL} pits on Lap {RIVAL_PIT}. {ME} tries to Overcut by pitting on Lap {MY_PIT}.")
print("-" * 50)

# Run the simulator
result = simulate_full_race(RACE, ME, RIVAL, target_pit_lap=MY_PIT, rival_pit_lap=RIVAL_PIT)

if result:
    my_time, rival_time, gap = result
    
    # Calculate position based on the gap
    points, position = calculate_points(gap)
    
    if gap > 0:
        print(f"✅ {ME} WINS by {abs(gap):.2f} seconds! (Position: {position})")
    else:
        print(f"❌ {ME} LOSES by {abs(gap):.2f} seconds. (Position: {position})")
        
    print(f"   {ME} Total Time: {my_time:.2f}s")
    print(f"   {RIVAL} Total Time: {rival_time:.2f}s")
else:
    print("Error: Could not find data for these drivers.")

🏁 Testing LEC vs NOR at British Grand Prix
Strategy: NOR pits on Lap 22. LEC tries to Overcut by pitting on Lap 28.
--------------------------------------------------
❌ LEC LOSES by 47.90 seconds. (Position: P2)
   LEC Total Time: 4668.39s
   NOR Total Time: 4620.49s


In [7]:
# Teammate Test: NOR vs PIA at Silverstone
RACE = "British Grand Prix"
ME = "NOR"
RIVAL = "PIA"
MY_PIT = 28     # NOR tries the Overcut
RIVAL_PIT = 22  # PIA pits early

print(f"🏁 Testing {ME} vs {RIVAL} at {RACE}")
print(f"Strategy: {RIVAL} pits on Lap {RIVAL_PIT}. {ME} tries to Overcut by pitting on Lap {MY_PIT}.")
print("-" * 50)

# Run the simulator
result = simulate_full_race(RACE, ME, RIVAL, target_pit_lap=MY_PIT, rival_pit_lap=RIVAL_PIT)

if result:
    my_time, rival_time, gap = result
    points, position = calculate_points(gap)
    
    if gap > 0:
        print(f"✅ {ME} WINS by {abs(gap):.2f} seconds! (Position: {position})")
    else:
        print(f"❌ {ME} LOSES by {abs(gap):.2f} seconds. (Position: {position})")
        
    print(f"   {ME} Total Time: {my_time:.2f}s")
    print(f"   {RIVAL} Total Time: {rival_time:.2f}s")
else:
    print("Error: Could not find data for these drivers.")

🏁 Testing NOR vs PIA at British Grand Prix
Strategy: PIA pits on Lap 22. NOR tries to Overcut by pitting on Lap 28.
--------------------------------------------------
❌ NOR LOSES by 26.29 seconds. (Position: P2)
   NOR Total Time: 4623.72s
   PIA Total Time: 4597.43s


# -------------------- H5: The Reactive Rival --------------------


In [8]:
def simulate_reactive_race(race_name, driver_code, rival_code, target_pit_lap, rival_initial_pit, pit_loss=22.0):
    my_data = DeepSeek_df[(DeepSeek_df['EventName'] == race_name) & (DeepSeek_df['Driver'] == driver_code)].copy()
    rival_data = DeepSeek_df[(DeepSeek_df['EventName'] == race_name) & (DeepSeek_df['Driver'] == rival_code)].copy()
    
    if my_data.empty or rival_data.empty: return None
    total_laps = int(max(my_data['LapNumber'].max(), rival_data['LapNumber'].max()))
    
    all_laps = pd.DataFrame({'LapNumber': range(1, total_laps + 1)})
    my_data = all_laps.merge(my_data, on='LapNumber', how='left').ffill()
    rival_data = all_laps.merge(rival_data, on='LapNumber', how='left').ffill()

    my_base_pace = my_data['LapTimeSeconds'].min()
    rival_base_pace = rival_data['LapTimeSeconds'].min()
    
    my_time, rival_time = 0.0, 0.0
    my_wear, rival_wear = 0.0, 0.0
    my_tyre_age, rival_tyre_age = 0, 0
    gap = 1.0 
    
    # --- H5 VARIABLES ---
    rival_pit_lap = rival_initial_pit # The rival's original plan
    rival_has_pitted = False
    my_has_pitted = False

    for lap in range(1, total_laps + 1):
        # 1. MY CAR PITS (The Trigger)
        if lap == target_pit_lap and not my_has_pitted:
            my_has_pitted = True
            my_pit_lap_actual = lap
            
            # --- H5: THE RIVAL REACTS ---
            # If we pit, and the rival hasn't pitted yet, and they were planning to pit later...
            if not rival_has_pitted and rival_pit_lap > lap:
                print(f"♟️ [Lap {lap}] We pit! Rival sees the Undercut threat.")
                print(f"♟️ [Lap {lap+1}] RIVAL REACTS: They change their plan and pit to cover us!")
                rival_pit_lap = lap + 1 # Force them to pit the next lap

        # 2. CALCULATE MY PACE
        my_row = my_data[my_data['LapNumber'] == lap].iloc[0]
        my_features = pd.DataFrame([{
            'TyreLife': my_tyre_age, 'TyreLifePercent': my_tyre_age / 25.0,
            'Avg_TrackTemp': my_row['Avg_TrackTemp'], 'Avg_AirTemp': my_row['Avg_AirTemp'],
            'TrackTemp_Delta': my_row['TrackTemp_Delta'], 'Peak_TyreEnergy': my_row['Peak_TyreEnergy'],
            'Energy_Std': my_row['Energy_Std'], 'PaceTrend': my_row['PaceTrend'],
            'Quad_Term': my_row['Quad_Term'], 'DegAcceleration': my_row['DegAcceleration'],
            'Compound': my_row['Compound'], 'CircuitArchetype': my_row['CircuitArchetype'],
            'StintPhase': 'Early' if my_tyre_age <= 5 else ('Mid' if my_tyre_age <= 15 else 'Late')
        }])
        my_wear_next = max(0, model.predict(my_features)[0])
        my_lap_time = my_base_pace + my_wear

        # 3. CALCULATE RIVAL PACE
        rival_row = rival_data[rival_data['LapNumber'] == lap].iloc[0]
        rival_features = pd.DataFrame([{
            'TyreLife': rival_tyre_age, 'TyreLifePercent': rival_tyre_age / 25.0,
            'Avg_TrackTemp': rival_row['Avg_TrackTemp'], 'Avg_AirTemp': rival_row['Avg_AirTemp'],
            'TrackTemp_Delta': rival_row['TrackTemp_Delta'], 'Peak_TyreEnergy': rival_row['Peak_TyreEnergy'],
            'Energy_Std': rival_row['Energy_Std'], 'PaceTrend': rival_row['PaceTrend'],
            'Quad_Term': rival_row['Quad_Term'], 'DegAcceleration': rival_row['DegAcceleration'],
            'Compound': rival_row['Compound'], 'CircuitArchetype': rival_row['CircuitArchetype'],
            'StintPhase': 'Early' if rival_tyre_age <= 5 else ('Mid' if rival_tyre_age <= 15 else 'Late')
        }])
        rival_wear_next = max(0, model.predict(rival_features)[0])
        rival_lap_time = rival_base_pace + rival_wear
        
        # 4. APPLY PIT STOPS
        if lap == rival_pit_lap and not rival_has_pitted:
            rival_has_pitted = True
            rival_lap_time += pit_loss
            rival_tyre_age = 1
            rival_wear = 0.0
        else:
            rival_tyre_age += 1
            rival_wear += rival_wear_next

        if lap == target_pit_lap and lap == my_pit_lap_actual:
            my_lap_time += pit_loss
            my_tyre_age = 1
            my_wear = 0.0
        else:
            my_tyre_age += 1
            my_wear += my_wear_next
            
        if 0 < gap < 1.5:
            my_lap_time += 0.3 
            
        my_time += my_lap_time
        rival_time += rival_lap_time
        gap = rival_time - my_time 

    return my_time, rival_time, gap

# ==========================================
# RUNNING THE H5 REACTIVE TEST
# ==========================================
RACE = "Bahrain Grand Prix"
ME = "HAM"
RIVAL = "VER"

print("🏁 H5 Test: We try to Undercut Verstappen.")
print(f"Original Plan: We pit on Lap 20. {RIVAL} plans to pit on Lap 30.")
print("-" * 60)

# We pit on Lap 20. VER originally wants to pit on Lap 30.
result = simulate_reactive_race(RACE, ME, RIVAL, target_pit_lap=20, rival_initial_pit=30)

if result:
    my_time, rival_time, gap = result
    if gap > 0:
        print(f"\n✅ We beat the reactive rival by {abs(gap):.2f} seconds!")
    else:
        print(f"\n❌ The rival's reaction worked. We lost by {abs(gap):.2f} seconds.")

🏁 H5 Test: We try to Undercut Verstappen.
Original Plan: We pit on Lap 20. VER plans to pit on Lap 30.
------------------------------------------------------------
♟️ [Lap 20] We pit! Rival sees the Undercut threat.
♟️ [Lap 21] RIVAL REACTS: They change their plan and pit to cover us!

❌ The rival's reaction worked. We lost by 4.88 seconds.


In [9]:
def simulate_bahrain_2025(race_name, me, rival):
    my_data = DeepSeek_df[(DeepSeek_df['EventName'] == race_name) & (DeepSeek_df['Driver'] == me)].copy()
    rival_data = DeepSeek_df[(DeepSeek_df['EventName'] == race_name) & (DeepSeek_df['Driver'] == rival)].copy()
    
    if my_data.empty or rival_data.empty: return None
    total_laps = int(max(my_data['LapNumber'].max(), rival_data['LapNumber'].max()))
    
    all_laps = pd.DataFrame({'LapNumber': range(1, total_laps + 1)})
    my_data = all_laps.merge(my_data, on='LapNumber', how='left').ffill()
    rival_data = all_laps.merge(rival_data, on='LapNumber', how='left').ffill()

    my_base_pace = my_data['LapTimeSeconds'].min()
    rival_base_pace = rival_data['LapTimeSeconds'].min()
    
    my_time, rival_time = 0.0, 0.0
    my_wear, rival_wear = 0.0, 0.0
    my_tyre_age, rival_tyre_age = 0, 0
    gap = 0.0 
    
    # HAM Strategy: Pit L17 and L32
    my_pit_laps = [17, 32]
    # VER Strategy: Pit L11 and L26
    rival_pit_laps = [11, 26]
    
    for lap in range(1, total_laps + 1):
        # --- MY CAR (HAM) ---
        my_row = my_data[my_data['LapNumber'] == lap].iloc[0]
        my_features = pd.DataFrame([{
            'TyreLife': my_tyre_age, 'TyreLifePercent': my_tyre_age / 25.0,
            'Avg_TrackTemp': my_row['Avg_TrackTemp'], 'Avg_AirTemp': my_row['Avg_AirTemp'],
            'TrackTemp_Delta': my_row['TrackTemp_Delta'], 'Peak_TyreEnergy': my_row['Peak_TyreEnergy'],
            'Energy_Std': my_row['Energy_Std'], 'PaceTrend': my_row['PaceTrend'],
            'Quad_Term': my_row['Quad_Term'], 'DegAcceleration': my_row['DegAcceleration'],
            'Compound': my_row['Compound'], 'CircuitArchetype': my_row['CircuitArchetype'],
            'StintPhase': 'Early' if my_tyre_age <= 5 else ('Mid' if my_tyre_age <= 15 else 'Late')
        }])
        my_wear_next = max(0, model.predict(my_features)[0])
        my_lap_time = my_base_pace + my_wear

        # --- RIVAL CAR (VER) ---
        rival_row = rival_data[rival_data['LapNumber'] == lap].iloc[0]
        rival_features = pd.DataFrame([{
            'TyreLife': rival_tyre_age, 'TyreLifePercent': rival_tyre_age / 25.0,
            'Avg_TrackTemp': rival_row['Avg_TrackTemp'], 'Avg_AirTemp': rival_row['Avg_AirTemp'],
            'TrackTemp_Delta': rival_row['TrackTemp_Delta'], 'Peak_TyreEnergy': rival_row['Peak_TyreEnergy'],
            'Energy_Std': rival_row['Energy_Std'], 'PaceTrend': rival_row['PaceTrend'],
            'Quad_Term': rival_row['Quad_Term'], 'DegAcceleration': rival_row['DegAcceleration'],
            'Compound': rival_row['Compound'], 'CircuitArchetype': rival_row['CircuitArchetype'],
            'StintPhase': 'Early' if rival_tyre_age <= 5 else ('Mid' if rival_tyre_age <= 15 else 'Late')
        }])
        rival_wear_next = max(0, model.predict(rival_features)[0])
        rival_lap_time = rival_base_pace + rival_wear
        
        # --- PIT STOPS ---
        # The standard pit loss is 22s, but HAM pits on L32 under SC (10s loss)
        my_pit_loss = 10.0 if lap == 32 else 22.0
        
        if lap in rival_pit_laps:
            rival_lap_time += 22.0
            rival_tyre_age = 1
            rival_wear = 0.0
        else:
            rival_tyre_age += 1
            rival_wear += rival_wear_next

        if lap in my_pit_laps:
            my_lap_time += my_pit_loss
            my_tyre_age = 1
            my_wear = 0.0
        else:
            my_tyre_age += 1
            my_wear += my_wear_next
            
        # Dirty Air Trap (Bahrain = 1.0)
        if 0 < gap < 1.5:
            my_lap_time += 0.3 
            
        my_time += my_lap_time
        rival_time += rival_lap_time
        gap = rival_time - my_time 

    return my_time, rival_time, gap

# ==========================================
# RUNNING THE 2025 BAHRAIN GP SIMULATION
# ==========================================
RACE = "Bahrain Grand Prix"
ME = "HAM"
RIVAL = "VER"

print(f"🏁 Simulating real 2025 Bahrain GP Strategies: {ME} vs {RIVAL}")
print(f"{ME} Strategy: Pits L17, L32 (SC) | {RIVAL} Strategy: Pits L11, L26")
print("-" * 60)

result = simulate_bahrain_2025(RACE, ME, RIVAL)

if result:
    my_time, rival_time, gap = result
    if gap > 0:
        print(f"✅ Simulator Result: {ME} beats {RIVAL} by {abs(gap):.2f} seconds!")
    else:
        print(f"❌ Simulator Result: {ME} loses to {RIVAL} by {abs(gap):.2f} seconds.")

🏁 Simulating real 2025 Bahrain GP Strategies: HAM vs VER
HAM Strategy: Pits L17, L32 (SC) | VER Strategy: Pits L11, L26
------------------------------------------------------------
✅ Simulator Result: HAM beats VER by 11.16 seconds!


In [10]:
import pandas as pd
import random

# ==========================================
# 1. THE MULTI-STOP SIMULATOR
# ==========================================
def simulate_multi_stop_race(race_name, driver_code, rival_code, my_pit_laps, rival_pit_laps, rival_penalty=0.0, pit_loss=22.0):
    my_data = DeepSeek_df[(DeepSeek_df['EventName'] == race_name) & (DeepSeek_df['Driver'] == driver_code)].copy()
    rival_data = DeepSeek_df[(DeepSeek_df['EventName'] == race_name) & (DeepSeek_df['Driver'] == rival_code)].copy()
    
    if my_data.empty or rival_data.empty: return None
    total_laps = int(max(my_data['LapNumber'].max(), rival_data['LapNumber'].max()))
    
    all_laps = pd.DataFrame({'LapNumber': range(1, total_laps + 1)})
    my_data = all_laps.merge(my_data, on='LapNumber', how='left').ffill()
    rival_data = all_laps.merge(rival_data, on='LapNumber', how='left').ffill()

    my_base_pace = my_data['LapTimeSeconds'].min()
    rival_base_pace = rival_data['LapTimeSeconds'].min()
    
    my_time, rival_time = 0.0, 0.0
    my_wear, rival_wear = 0.0, 0.0
    my_tyre_age, rival_tyre_age = 0, 0
    gap = 0.0 
    
    for lap in range(1, total_laps + 1):
        # MY CAR
        my_row = my_data[my_data['LapNumber'] == lap].iloc[0]
        my_features = pd.DataFrame([{
            'TyreLife': my_tyre_age, 'TyreLifePercent': my_tyre_age / 25.0,
            'Avg_TrackTemp': my_row['Avg_TrackTemp'], 'Avg_AirTemp': my_row['Avg_AirTemp'],
            'TrackTemp_Delta': my_row['TrackTemp_Delta'], 'Peak_TyreEnergy': my_row['Peak_TyreEnergy'],
            'Energy_Std': my_row['Energy_Std'], 'PaceTrend': my_row['PaceTrend'],
            'Quad_Term': my_row['Quad_Term'], 'DegAcceleration': my_row['DegAcceleration'],
            'Compound': my_row['Compound'], 'CircuitArchetype': my_row['CircuitArchetype'],
            'StintPhase': 'Early' if my_tyre_age <= 5 else ('Mid' if my_tyre_age <= 15 else 'Late')
        }])
        my_wear_next = max(0, model.predict(my_features)[0])
        my_lap_time = my_base_pace + my_wear

        # RIVAL CAR
        rival_row = rival_data[rival_data['LapNumber'] == lap].iloc[0]
        rival_features = pd.DataFrame([{
            'TyreLife': rival_tyre_age, 'TyreLifePercent': rival_tyre_age / 25.0,
            'Avg_TrackTemp': rival_row['Avg_TrackTemp'], 'Avg_AirTemp': rival_row['Avg_AirTemp'],
            'TrackTemp_Delta': rival_row['TrackTemp_Delta'], 'Peak_TyreEnergy': rival_row['Peak_TyreEnergy'],
            'Energy_Std': rival_row['Energy_Std'], 'PaceTrend': rival_row['PaceTrend'],
            'Quad_Term': rival_row['Quad_Term'], 'DegAcceleration': rival_row['DegAcceleration'],
            'Compound': rival_row['Compound'], 'CircuitArchetype': rival_row['CircuitArchetype'],
            'StintPhase': 'Early' if rival_tyre_age <= 5 else ('Mid' if rival_tyre_age <= 15 else 'Late')
        }])
        rival_wear_next = max(0, model.predict(rival_features)[0])
        rival_lap_time = rival_base_pace + rival_wear
        
        # PIT STOPS
        my_pit_loss = 10.0 if lap == 32 else 22.0 # HAM gets cheap SC pit on L32
        
        if lap in rival_pit_laps:
            rival_lap_time += pit_loss + rival_penalty
            rival_tyre_age = 1
            rival_wear = 0.0
        else:
            rival_tyre_age += 1
            rival_wear += rival_wear_next

        if lap in my_pit_laps:
            my_lap_time += my_pit_loss
            my_tyre_age = 1
            my_wear = 0.0
        else:
            my_tyre_age += 1
            my_wear += my_wear_next
            
        # DIRTY AIR
        if 0 < gap < 1.5:
            my_lap_time += 0.3 
            
        my_time += my_lap_time
        rival_time += rival_lap_time
        gap = rival_time - my_time 

    return my_time, rival_time, gap

# ==========================================
# 2. THE SWEEPER: Finding VER's winning strategy
# ==========================================
RACE = "Bahrain Grand Prix"
ME = "VER"    
RIVAL = "HAM" 

HAM_PITS = [17, 32] # Hamilton's locked real strategy

print(f"🏁 Finding the winning strategy for {ME} against {RIVAL} at {RACE}")
print(f"Locked {RIVAL} Strategy: {HAM_PITS}")
print("-" * 60)

best_lap = 0
best_gap = -999

# Sweep VER's pit lap from 20 to 35
for my_pit in range(20, 36):
    result = simulate_multi_stop_race(
        RACE, ME, RIVAL, 
        my_pit_laps=[my_pit],       
        rival_pit_laps=HAM_PITS,    
        rival_penalty=0.0           
    )
    
    if result:
        my_time, rival_time, gap = result
        
        if gap > 0:
            print(f"✅ If VER pits on Lap {my_pit}, VER WINS by {abs(gap):.2f} seconds!")
            
            if gap > best_gap:
                best_gap = gap
                best_lap = my_pit

print("-" * 60)
if best_gap > -999:
    print(f"🏆 THE WINNING STRATEGY FOR VER: Pit on Lap {best_lap} (Wins by {best_gap:.2f}s)")
else:
    print("❌ No 1-stop strategy beats HAM's 2-stop.")

🏁 Finding the winning strategy for VER against HAM at Bahrain Grand Prix
Locked HAM Strategy: [17, 32]
------------------------------------------------------------
✅ If VER pits on Lap 20, VER WINS by 10.68 seconds!
✅ If VER pits on Lap 21, VER WINS by 10.07 seconds!
✅ If VER pits on Lap 22, VER WINS by 11.07 seconds!
✅ If VER pits on Lap 23, VER WINS by 11.73 seconds!
✅ If VER pits on Lap 24, VER WINS by 12.48 seconds!
✅ If VER pits on Lap 25, VER WINS by 13.15 seconds!
✅ If VER pits on Lap 26, VER WINS by 13.74 seconds!
✅ If VER pits on Lap 27, VER WINS by 14.07 seconds!
✅ If VER pits on Lap 28, VER WINS by 12.95 seconds!
✅ If VER pits on Lap 29, VER WINS by 13.18 seconds!
✅ If VER pits on Lap 30, VER WINS by 13.24 seconds!
✅ If VER pits on Lap 31, VER WINS by 13.05 seconds!
✅ If VER pits on Lap 32, VER WINS by 24.80 seconds!
✅ If VER pits on Lap 33, VER WINS by 12.63 seconds!
✅ If VER pits on Lap 34, VER WINS by 12.35 seconds!
✅ If VER pits on Lap 35, VER WINS by 11.94 seconds!
----

# -------------------- H5: The Reactive Rival --------------------


# -------------------- H6: The MASTER SIMULATOR CODE --------------------


In [18]:
# ==========================================
# H6: THE DECISION ENGINE (Strategy Optimizer)
# ==========================================
def evaluate_strategies(race_name, driver, rival, rival_pit_laps, plans, num_sims=20):
    print(f"==================================================")
    print(f"🏁 STRATEGY REPORT: {race_name} - {driver}")
    print(f"Rival Strategy Locked ({rival}): {rival_pit_laps}")
    print(f"==================================================")
    
    best_plan = None
    best_win_rate = -1
    best_avg_gap = -999
    
    for plan_name, my_pits in plans.items():
        wins = 0
        total_gap = 0
        
        # H3: Monte Carlo (Run 20 times to factor in random Safety Cars)
        for _ in range(num_sims):
            result = simulate_master_race(
                race_name, driver, rival, 
                my_pit_laps=my_pits, 
                rival_pit_laps=rival_pit_laps.copy(),
                reactive_rival=True # H5: Rival reacts to us
            )
            if result:
                my_time, rival_time, gap = result
                total_gap += gap
                if gap > 0:
                    wins += 1
                    
        win_rate = (wins / num_sims) * 100
        avg_gap = total_gap / num_sims
        
        # H1: Objective (Points based on win rate)
        expected_points = (win_rate * 0.01 * 25) + ((100 - win_rate) * 0.01 * 18)
        
        # H4: Track Context (Explain the Dirty Air / Multiplier)
        if "Monaco" in race_name:
            track_note = "High Dirty Air penalty applied (Hard to pass)."
        elif "Monza" in race_name or "Italy" in race_name:
            track_note = "Low Dirty Air penalty applied (Easy to pass)."
        else:
            track_note = "Standard Dirty Air penalty applied."

        # Output the breakdown for this plan
        print(f"▶️ EVALUATING {plan_name}: Pits on {my_pits}")
        print(f"   - H1 (Objective): Expected {expected_points:.1f} Points.")
        print(f"   - H3 (Risk/Prob): Win Rate: {win_rate:.0f}% | Avg Gap: {avg_gap:+.2f}s")
        print(f"   - H4 (Track): {track_note}")
        print(f"   - H5 (Rival): Rival covered our undercut.")
        print("-" * 50)
        
        # Save the best plan
        if win_rate > best_win_rate:
            best_win_rate = win_rate
            best_plan = plan_name
            best_avg_gap = avg_gap

    # H6: THE FINAL DECISION
    print(f"🏆 DECISION ENGINE (H6):")
    print(f"Recommendation: GO WITH {best_plan}.")
    print(f"Probability of Winning: {best_win_rate:.0f}%")
    if best_avg_gap > 0:
        print(f"Reasoning: This strategy beats the rival by an average of {best_avg_gap:.2f}s across {num_sims} simulations.")
    else:
        print(f"Reasoning: All strategies lose, but {best_plan} minimizes the damage (avg loss: {abs(best_avg_gap):.2f}s).")
    print("==================================================")


# ==========================================
# RUNNING THE H6 TEST
# ==========================================
RACE = "Bahrain Grand Prix"
ME = "HAM"
RIVAL = "VER"

# The Rival's Strategy (We assume VER pits L25)
RIVAL_PITS = [25]

# Our 3 Plans to Test
PLANS = {
    "PLAN A (Undercut)": [24],
    "PLAN B (Same)": [25],
    "PLAN C (Overcut)": [26]
}

# Run the H6 Engine!
evaluate_strategies(RACE, ME, RIVAL, RIVAL_PITS, PLANS, num_sims=20)

🏁 STRATEGY REPORT: Bahrain Grand Prix - HAM
Rival Strategy Locked (VER): [25]
▶️ EVALUATING PLAN A (Undercut): Pits on []
   - H1 (Objective): Expected 18.0 Points.
   - H3 (Risk/Prob): Win Rate: 0% | Avg Gap: -14.24s
   - H4 (Track): Standard Dirty Air penalty applied.
   - H5 (Rival): Rival covered our undercut.
--------------------------------------------------
▶️ EVALUATING PLAN B (Same): Pits on []
   - H1 (Objective): Expected 18.0 Points.
   - H3 (Risk/Prob): Win Rate: 0% | Avg Gap: -14.17s
   - H4 (Track): Standard Dirty Air penalty applied.
   - H5 (Rival): Rival covered our undercut.
--------------------------------------------------
▶️ EVALUATING PLAN C (Overcut): Pits on []
   - H1 (Objective): Expected 18.0 Points.
   - H3 (Risk/Prob): Win Rate: 0% | Avg Gap: -14.15s
   - H4 (Track): Standard Dirty Air penalty applied.
   - H5 (Rival): Rival covered our undercut.
--------------------------------------------------
🏆 DECISION ENGINE (H6):
Recommendation: GO WITH PLAN A (Unde